# Customer Churn Prediction — EDA & Modeling

This notebook explores the Telco Customer Churn dataset and walks through the same preprocessing and modeling pipeline used in `src/`. Use it to explore the data interactively; use the scripts in `src/` for reproducible training.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()

## 1. Basic overview

In [ ]:
print(df.shape)
df.info()

In [ ]:
df["Churn"].value_counts(normalize=True).plot(kind="bar", title="Churn distribution")

## 2. Clean TotalCharges (stored as text, has blanks for tenure=0 customers)

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

## 3. Churn vs. key features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[2])
axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout()

Customers with month-to-month contracts, low tenure, and high monthly charges churn far more often.

## 4. Reuse the project's preprocessing pipeline

In [ ]:
import sys
sys.path.append("../src")
from data_preprocessing import preprocess_pipeline

import os
os.chdir("..")  # so relative paths (data/, models/) resolve like the scripts expect

X_train, X_test, y_train, y_test = preprocess_pipeline()
X_train.shape, X_test.shape

## 5. Train and compare models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    print(name, "ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

## 6. Next steps

For the full training run (all 6 candidate models, saved artifacts, plots), use:
```bash
python src/train_model.py
```
This saves the best model to `models/churn_model.pkl` along with evaluation plots in `outputs/`.